# Colab runner: WESAD wrist-only LOSO with the `stressnet` package

Thin notebook: all logic lives in `src/stressnet`. Upload the repo folder (without `.venv`, `data/`, `cache/`)
to Drive at `MyDrive/licenta/code`, and keep the WESAD zip at `MyDrive/licenta/WESAD/WESAD.zip`.
Runtime > Change runtime type > GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q -e /content/drive/MyDrive/licenta/code
!unzip -q -n "/content/drive/MyDrive/licenta/WESAD/WESAD.zip" -d /content/wesad_data

In [ ]:
from stressnet.constants import LABEL_MAP, WRIST_MODALITIES
from stressnet.data import wesad
from stressnet.data.quality import apply_gate
from stressnet.data.splits import to_binary_windows
from stressnet.training import get_device, loso_deep_cv
from stressnet.evaluation import summarize_subsets

DEVICE = get_device()
CACHE = "/content/drive/MyDrive/licenta/WESAD/wesad_cache"

## Windows (wrist-only storage, chest-based clip flags still computed)
`keep_chest=False` keeps the cache around 100 MB instead of several GB, and the build takes a few minutes.

In [ ]:
windows, report = wesad.load_or_build_final_dataset(
    f"{CACHE}/wesad_wrist_windows.pkl", "/content/wesad_data/WESAD", LABEL_MAP, keep_chest=False)
final_windows, _ = apply_gate(windows, clip=True)
final_windows_bin = to_binary_windows(final_windows)
print(len(final_windows_bin), "windows")

## Deep LOSO with modality dropout (resumable: finished folds are cached on Drive)

In [ ]:
MODALITY_DROPOUT_P = {"wrist_bvp": 0.05, "wrist_eda": 0.05, "wrist_acc": 0.4, "wrist_temp": 0.4}
EVAL_SUBSETS = {
    "bvp_eda_only": {"wrist_acc": False, "wrist_temp": False},
    "no_temp":      {"wrist_temp": False},
}

results = {}
for body in ("cnn", "tcn"):
    results[body] = loso_deep_cv(
        WRIST_MODALITIES, body, "wrist_binary_dropout", final_windows_bin,
        num_classes=2, epochs=20, modality_dropout_p=MODALITY_DROPOUT_P, eval_subsets=EVAL_SUBSETS,
        cache_dir=f"{CACHE}/binary_loso_dropout", select_best=False, device=DEVICE)
    display(summarize_subsets(results[body]))